# Notebook 2 — Introduction to Image Classification

**Use case:** Manufacturing Visual Quality Inspection

**What you will do:**
1. Load the pre-trained YOLOv8 model from the git repo (no internet download).
2. Run it against the 4 real PCB defect images that come with the repo.
3. Read the confidence scores and understand the raw output.
4. See *why* fine-tuning is needed — the pre-trained model has never seen PCB defects.

| Image file | What it shows |
|---|---|
| `pass.png` | A clean, defect-free PCB — should be approved |
| `scratch.jpg` | A PCB with a surface scratch cutting through traces |
| `crack.jpg` | A blown/cracked solder joint on a PCB mounting hole |
| `contamination.jpg` | A PCB flooded with liquid/flux contamination |

> **Note:** The pre-trained model knows 1000 ImageNet classes — not PCB defect categories.
> After this notebook you will see it predicts random/wrong labels for all 4 images.
> Notebook 3 fixes this by fine-tuning on your defect dataset.

In [ ]:
# ── Cell 0a: Install missing packages ─────────────────────────────────────────
# ultralytics and onnxruntime are not pre-installed in the RHOAI workbench image.
# Safe to re-run — pip skips packages that are already installed.
import subprocess, sys

PACKAGES = ['ultralytics==8.4.96', 'onnxruntime==1.21.0']

print('Installing required packages (first run ~60 s, subsequent runs instant)...')
for pkg in PACKAGES:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-cache-dir', pkg],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print(f'  ✅ {pkg}')
    else:
        print(f'  ❌ {pkg} FAILED — {r.stderr[-300:]}')
print('\n✅ Package install complete')

In [ ]:
# ── Cell 0b: Sync lab materials from GitHub ───────────────────────────────────
import subprocess, os, pathlib

REPO_URL   = 'https://github.com/faheemshai/1512_model_training.git'
LOCAL_PATH = os.path.expanduser('~/lab-materials')

if pathlib.Path(LOCAL_PATH, '.git').is_dir():
    r = subprocess.run(['git', '-C', LOCAL_PATH, 'pull', '--ff-only'],
                       capture_output=True, text=True)
    print('Repo up to date:', r.stdout.strip() or 'Already up to date.')
else:
    print('Cloning lab repo (first time, ~10 s)...')
    r = subprocess.run(['git', 'clone', REPO_URL, LOCAL_PATH],
                       capture_output=True, text=True)
    print(r.stderr.strip())

LAB = LOCAL_PATH
print(f'✅ Lab materials ready at: {LAB}')

In [ ]:
from ultralytics import YOLO
from PIL import Image
import pathlib

# ── Load weights from the git repo — no internet download needed ──────────────
WEIGHTS = str(pathlib.Path(LAB) / 'models' / 'yolov8n-cls.pt')
model   = YOLO(WEIGHTS)
print(f'✅ YOLOv8n-cls loaded from {WEIGHTS}')
print(f'   ImageNet-1k classes: {len(model.names)}')
print('   ⚠️  None of these 1000 classes are PCB defect categories — fine-tuning needed!')

In [ ]:
# ── Verify the 4 PCB sample images are present ────────────────────────────────
IMG_DIR = pathlib.Path(LAB) / 'sample-images'

SAMPLES = ['pass.png',      'scratch.jpg',               'crack.jpg',                  'contamination.jpg']
LABELS  = ['Clean PCB (PASS)', 'Scratch on traces (FAIL)', 'Blown solder joint (FAIL)', 'Liquid contamination (FAIL)']

all_present = True
for fname, label in zip(SAMPLES, LABELS):
    path = IMG_DIR / fname
    if path.exists():
        print(f'  ✅ {fname:<22}  {path.stat().st_size:>8,} bytes  — {label}')
    else:
        print(f'  ❌ {fname:<22}  NOT FOUND at {path}')
        all_present = False

if all_present:
    print('\n✅ All 4 images present — ready to classify')
else:
    raise FileNotFoundError('Sample images missing — re-run the git clone cell above.')

In [ ]:
# ── Run pre-trained model on each image and print top-5 scores ────────────────
#
# OBSERVATION: The model will NOT predict 'pass', 'scratch', 'crack', or 'contamination'.
# It predicts ImageNet classes like 'circuit_board', 'hook', 'printed_circuit' etc.
# This demonstrates WHY fine-tuning is required before deployment.

print('Pre-trained YOLOv8n-cls predictions (ImageNet classes — NOT defect classes):\n')
print('─' * 70)

for fname, label in zip(SAMPLES, LABELS):
    img_path = str(IMG_DIR / fname)
    results  = model(img_path, verbose=False)
    r        = results[0]

    top5_idx   = r.probs.top5
    top5_conf  = r.probs.top5conf.tolist()
    top5_names = [r.names[i] for i in top5_idx]

    print(f'\n📷  {label}')
    print(f'    File: {fname}')
    print(f'    Top ImageNet prediction: "{top5_names[0]}" ({top5_conf[0]*100:.1f}%)')
    print(f'    ❌ Expected one of: pass / scratch / crack / contamination')
    print('    Top-5 ImageNet guesses:')
    for name, conf in zip(top5_names, top5_conf):
        bar = '█' * int(conf * 25)
        print(f'      {name:<30} {bar:<25} {conf*100:5.1f}%')

print('\n' + '─' * 70)
print('\n📌 Key observation:')
print('   The pre-trained model has zero concept of PCB defects.')
print('   It maps everything to generic ImageNet categories.')
print('   ➡  Notebook 3 will fine-tune it to classify defects correctly.')

In [ ]:
# ── Visualise all 4 images side-by-side with their top-1 predictions ──────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, fname, label in zip(axes, SAMPLES, LABELS):
    img_path = str(IMG_DIR / fname)
    results  = model(img_path, verbose=False)
    r        = results[0]
    top1_name = r.names[r.probs.top1]
    top1_conf = r.probs.top1conf.item()

    ax.imshow(mpimg.imread(img_path))
    ax.set_title(
        f'{label}\n→ ImageNet: "{top1_name}"\n({top1_conf*100:.1f}%)',
        fontsize=9, pad=8, color='#c00'
    )
    ax.axis('off')

plt.suptitle(
    'Pre-trained YOLOv8n-cls — ImageNet predictions (BEFORE fine-tuning)\n'
    'None of these are valid defect classifications',
    fontsize=11, y=1.03, color='#333'
)
plt.tight_layout()
plt.show()

print('\n✅ Notebook 2 complete — proceed to Notebook 3 to fine-tune the defect classifier')